<a href="https://colab.research.google.com/github/manaswiarvind/bis-lab/blob/main/WEEK1%20_1BF24CS165_GENETIC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import pandas as pd

# -----------------------------------------
# 1. INITIALIZE EXAMINATION DATA
# -----------------------------------------

subjects = [
    "DSA", "DBMS", "OS", "CN",
    "OOP", "SE", "AI", "Maths"
]

students = {
    "DSA":  ["S1", "S2", "S3", "S4"],
    "DBMS": ["S1", "S2", "S5", "S6"],
    "OS":   ["S3", "S4", "S5", "S7"],
    "CN":   ["S1", "S6", "S7", "S8"],
    "OOP":  ["S2", "S3", "S6", "S8"],
    "SE":   ["S4", "S5", "S7", "S8"],
    "AI":   ["S1", "S3", "S5", "S8"],
    "Maths":["S2", "S4", "S6", "S7"]
}

faculty = {
    "DSA": "F1",
    "DBMS": "F2",
    "OS": "F3",
    "CN": "F1",
    "OOP": "F2",
    "SE": "F4",
    "AI": "F3",
    "Maths": "F4"
}

classrooms = {
    "Room A": 5,
    "Room B": 7,
    "Room C": 10
}

# Number of examination time slots
time_slots = [
    "9 AM - 11 AM",
    "11 AM - 1 PM",
    "1 PM - 3 PM",
    "3 PM - 5 PM",
    "5 PM - 7 PM"
]

# Students appearing for each subject
enrollment = {
    subject: len(students[subject])
    for subject in subjects
}

# Allowed examination slots for each subject
# Empty restriction means all slots are allowed
allowed_slots = {
    subject: list(range(len(time_slots)))
    for subject in subjects
}


# -----------------------------------------
# 2. GENETIC ALGORITHM PARAMETERS
# -----------------------------------------

POPULATION_SIZE = 30
MAX_GENERATIONS = 100
MUTATION_PROBABILITY = 0.20

# Each chromosome contains:
# (Time slot, Classroom) for every subject


# -----------------------------------------
# 3. GENERATE RANDOM TIMETABLE
# -----------------------------------------

def generate_timetable():

    timetable = []

    for subject in subjects:

        slot = random.choice(allowed_slots[subject])

        room = random.choice(list(classrooms.keys()))

        timetable.append((slot, room))

    return timetable


# Generate initial population
population = [
    generate_timetable()
    for _ in range(POPULATION_SIZE)
]


# -----------------------------------------
# 4. CALCULATE PENALTY
# -----------------------------------------

def calculate_penalty(timetable):

    penalty = 0

    # Check every pair of examinations
    for i in range(len(subjects)):

        for j in range(i + 1, len(subjects)):

            sub1 = subjects[i]
            sub2 = subjects[j]

            slot1, room1 = timetable[i]
            slot2, room2 = timetable[j]

            # Student examination clashes
            if slot1 == slot2:

                common_students = (
                    set(students[sub1]) &
                    set(students[sub2])
                )

                penalty += len(common_students)

                # Faculty duty clashes
                if faculty[sub1] == faculty[sub2]:
                    penalty += 1

    # Classroom capacity violations
    for i, subject in enumerate(subjects):

        slot, room = timetable[i]

        if enrollment[subject] > classrooms[room]:
            penalty += 1

    # Invalid time slot assignments
    for i, subject in enumerate(subjects):

        slot, room = timetable[i]

        if slot not in allowed_slots[subject]:
            penalty += 1

    return penalty


# -----------------------------------------
# 5. FITNESS FUNCTION
# -----------------------------------------

def fitness(timetable):

    penalty = calculate_penalty(timetable)

    return 1 / (1 + penalty)


# -----------------------------------------
# 6. SELECT THE BEST TIMETABLES
# -----------------------------------------

def select_parent(population):

    selected = random.sample(population, 3)

    best = max(selected, key=fitness)

    return best


# -----------------------------------------
# 7. CROSSOVER
# -----------------------------------------

def crossover(parent1, parent2):

    point = random.randint(1, len(subjects) - 1)

    child = (
        parent1[:point] +
        parent2[point:]
    )

    return child


# -----------------------------------------
# 8. MUTATION
# -----------------------------------------

def mutation(timetable):

    timetable = timetable.copy()

    for i in range(len(subjects)):

        if random.random() < MUTATION_PROBABILITY:

            slot = random.choice(
                allowed_slots[subjects[i]]
            )

            room = random.choice(
                list(classrooms.keys())
            )

            timetable[i] = (slot, room)

    return timetable


# -----------------------------------------
# 9. GENETIC ALGORITHM EXECUTION
# -----------------------------------------

best_timetable = None
best_penalty = float('inf')

print("Genetic Algorithm Execution Started\n")

for generation in range(MAX_GENERATIONS):

    new_population = []

    # Preserve the best timetable (Elitism)
    elite = min(population, key=calculate_penalty)
    new_population.append(elite.copy())

    while len(new_population) < POPULATION_SIZE:

        # Select two parent timetables
        parent1 = select_parent(population)
        parent2 = select_parent(population)

        # Apply crossover
        child = crossover(parent1, parent2)

        # Apply mutation
        child = mutation(child)

        # Add child timetable to new population
        new_population.append(child)

    population = new_population

    # Find best timetable in current generation
    current_best = min(population, key=calculate_penalty)

    current_penalty = calculate_penalty(current_best)

    if current_penalty < best_penalty:

        best_penalty = current_penalty
        best_timetable = current_best.copy()

    # Display progress
    if generation % 10 == 0:

        print(
            "Generation:", generation,
            "| Best Penalty:", best_penalty
        )


# -----------------------------------------
# 10. DISPLAY OPTIMIZED TIMETABLE
# -----------------------------------------

print("\n===================================")
print(" OPTIMIZED EXAMINATION TIMETABLE")
print("===================================\n")

result = []

for i, subject in enumerate(subjects):

    slot, room = best_timetable[i]

    result.append({
        "Subject": subject,
        "Faculty": faculty[subject],
        "Time Slot": time_slots[slot],
        "Classroom": room,
        "Students": enrollment[subject]
    })

df = pd.DataFrame(result)

print(df.to_string(index=False))


# -----------------------------------------
# 11. DISPLAY FINAL FITNESS
# -----------------------------------------

print("\nFinal Penalty:", best_penalty)

print("Final Fitness:", round(fitness(best_timetable), 4))

if best_penalty == 0:

    print("\nSUCCESS: Valid examination timetable generated!")

else:

    print("\nBest possible timetable found within the given generations.")
    print("Some constraint violations still remain.")

Genetic Algorithm Execution Started

Generation: 0 | Best Penalty: 4
Generation: 10 | Best Penalty: 2
Generation: 20 | Best Penalty: 2
Generation: 30 | Best Penalty: 2
Generation: 40 | Best Penalty: 2
Generation: 50 | Best Penalty: 2
Generation: 60 | Best Penalty: 2
Generation: 70 | Best Penalty: 2
Generation: 80 | Best Penalty: 2
Generation: 90 | Best Penalty: 2

 OPTIMIZED EXAMINATION TIMETABLE

Subject Faculty    Time Slot Classroom  Students
    DSA      F1  5 PM - 7 PM    Room B         4
   DBMS      F2 9 AM - 11 AM    Room A         4
     OS      F3 9 AM - 11 AM    Room C         4
     CN      F1  1 PM - 3 PM    Room B         4
    OOP      F2  3 PM - 5 PM    Room A         4
     SE      F4  3 PM - 5 PM    Room B         4
     AI      F3 11 AM - 1 PM    Room B         4
  Maths      F4 11 AM - 1 PM    Room B         4

Final Penalty: 2
Final Fitness: 0.3333

Best possible timetable found within the given generations.
Some constraint violations still remain.
